# 02 DQA-SoftMoX Mix Weight Optimizer

Learn module-wise `G_t / A_t / S_t` mixing coefficients with a small black-box optimizer.

In [1]:
from __future__ import annotations

import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "dynamic_quality_aware_classwise_aggregation").exists():
    REPO_ROOT = Path("/app/Object_Detection")

JUDGER_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation" / "moe_dqa_judger"
RUNNER = JUDGER_ROOT / "scripts" / "run_02_mix_weight_optimizer.py"
WORKSPACE = JUDGER_ROOT / "output" / "02_mix_weight_optimizer"
LOG_DIR = JUDGER_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RUNNER:", RUNNER, RUNNER.exists())
print("WORKSPACE:", WORKSPACE)


REPO_ROOT: /app/Object_Detection
RUNNER: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_02_mix_weight_optimizer.py True
WORKSPACE: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/02_mix_weight_optimizer


## Design

In [2]:
design = pd.DataFrame(
    [
        {
            "part": "candidate models",
            "choice": "G_t / A_t / S_t",
            "reason": "Keep the previous global anchor, the DQA aggregate, and the server-repaired model as the only three ingredients.",
        },
        {
            "part": "granularity",
            "choice": "body / head / moe",
            "reason": "Approximate pFedLA-style layer-wise aggregation while keeping the search small enough for YOLO checkpoints.",
        },
        {
            "part": "learning signal",
            "choice": "mini total-val mAP surrogate",
            "reason": "AdaMerging/model-soup style black-box coefficient learning without adding an external teacher.",
        },
        {
            "part": "optimizer",
            "choice": "RF surrogate + Dirichlet local search",
            "reason": "FedAWA/FedLAW-like adaptive aggregation, but learned from observed validation response instead of fixed hand weights.",
        },
        {
            "part": "guardrail",
            "choice": "full total-val for top candidates",
            "reason": "Mini split is only for cheap search; the selected checkpoints are verified on the paper total split.",
        },
    ]
)
display(design)


,part,choice,reason
0,candidate models,G_t / A_t / S_t,"Keep the previous global anchor, the DQA aggre..."
1,granularity,body / head / moe,Approximate pFedLA-style layer-wise aggregatio...
2,learning signal,mini total-val mAP surrogate,AdaMerging/model-soup style black-box coeffici...
3,optimizer,RF surrogate + Dirichlet local search,"FedAWA/FedLAW-like adaptive aggregation, but l..."
4,guardrail,full total-val for top candidates,Mini split is only for cheap search; the selec...


## Round 1-2 Optimizer

This pass searches the first two rounds on a mini total split, then verifies top candidates on the full total split.

In [3]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log_path = LOG_DIR / f"02_mix_weight_optimizer_{timestamp}.log"
cmd = [
    sys.executable,
    str(RUNNER),
    "--workspace-root", str(WORKSPACE),
    "--rounds", "1,2",
    "--mini-images", "384",
    "--random-candidates", "6",
    "--surrogate-iterations", "2",
    "--surrogate-pool", "48",
    "--surrogate-evals", "3",
    "--full-eval-topk", "2",
    "--val-batch-size", "32",
    "--force",
]
print(" ".join(cmd))
with log_path.open("w", encoding="utf-8") as log:
    proc = subprocess.run(cmd, cwd=REPO_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log.write(proc.stdout)
print("returncode:", proc.returncode)
print("log:", log_path)
print(proc.stdout[-6000:])
if proc.returncode != 0:
    raise SystemExit(proc.returncode)


/opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_02_mix_weight_optimizer.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/02_mix_weight_optimizer --rounds 1,2 --mini-images 384 --random-candidates 6 --surrogate-iterations 2 --surrogate-pool 48 --surrogate-evals 3 --full-eval-topk 2 --val-batch-size 32 --force


returncode: 0
log: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/logs/02_mix_weight_optimizer_20260513_115535.log
18505338078,
    "expert_entropy_norm": 0.7377531204192945,
    "dead_expert_fraction": 0.25,
    "repair_gain_vs_a": 0.0033250000000001334,
    "repair_gain_vs_g": 0.007920000000000038,
    "path": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/02_mix_weight_optimizer/candidates/r001_best01_prior02.pt",
    "eval_scope": "full_total",
    "returncode": 0,
    "log_file": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/02_mix_weight_optimizer/validation_logs/r001_best01_prior02_full.log",
    "command": "/root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/navigating_data_heterogeneity/vendor/efficientteacher/val.py --weights /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/02_mix_weight_optimizer/candidates

## Results

In [4]:
trials_csv = WORKSPACE / "stats" / "02_mix_weight_optimizer_trials.csv"
best_csv = WORKSPACE / "stats" / "02_mix_weight_optimizer_best_full.csv"
trials = pd.read_csv(trials_csv)
best = pd.read_csv(best_csv)

display(best[[
    "round", "candidate_id", "map50", "map50_95", "precision", "recall", "score",
    "body_g", "body_a", "body_s", "head_g", "head_a", "head_s", "moe_g", "moe_a", "moe_s",
]].head(12))

mini_cols = [
    "round", "candidate_id", "phase", "map50", "map50_95", "precision", "recall", "score",
    "body_g", "body_a", "body_s", "head_g", "head_a", "head_s", "moe_g", "moe_a", "moe_s",
]
display(
    trials[trials["eval_scope"].eq("mini")]
    .sort_values(["round", "score"], ascending=[True, False])
    [mini_cols]
    .groupby("round")
    .head(8)
)
print("report:", WORKSPACE / "02_mix_weight_optimizer_report.md")


,round,candidate_id,map50,map50_95,precision,recall,score,body_g,body_a,body_s,head_g,head_a,head_s,moe_g,moe_a,moe_s
0,1.0,best00_sur01_00,0.462,0.26,0.694,0.431,0.57455,0.003656,5.507407e-10,0.996344,0.000117,3.389785e-19,0.999883,0.067663,0.002631,0.929707
1,1.0,best01_prior02,0.462,0.26,0.694,0.431,0.57455,0.000100,1.000000e-04,0.999800,0.000100,1.000000e-04,0.999800,0.000100,0.000100,0.999800
2,2.0,best00_sur00_02,0.462,0.26,0.718,0.420,0.57400,0.588569,2.807420e-01,0.130689,0.280795,6.562040e-02,0.653585,0.255120,0.662192,0.082688
3,2.0,best01_sur00_00,0.462,0.26,0.718,0.420,0.57400,0.617933,2.391548e-01,0.142912,0.168110,5.772729e-02,0.774163,0.124189,0.788428,0.087383


,round,candidate_id,phase,map50,map50_95,precision,recall,score,body_g,body_a,body_s,head_g,head_a,head_s,moe_g,moe_a,moe_s
20,1.0,sur01_00,surrogate1,0.546,0.304,0.768,0.483,0.67655,3.655741e-03,5.507407e-10,0.996344,1.172966e-04,3.389785e-19,0.999883,6.766251e-02,2.630946e-03,0.929707
3,1.0,prior02,init,0.546,0.304,0.779,0.479,0.67635,1.000000e-04,1.000000e-04,0.999800,1.000000e-04,1.000000e-04,0.999800,1.000000e-04,1.000000e-04,0.999800
17,1.0,sur00_00,surrogate0,0.546,0.304,0.779,0.479,0.67635,5.448302e-07,4.001623e-03,0.995998,2.829681e-05,1.452233e-04,0.999826,1.529539e-03,3.029875e-07,0.998470
18,1.0,sur00_01,surrogate0,0.546,0.304,0.779,0.479,0.67635,8.216980e-38,6.558879e-06,0.999993,1.856968e-09,1.458528e-03,0.998541,6.865422e-08,1.003076e-04,0.999900
21,1.0,sur01_01,surrogate1,0.546,0.304,0.779,0.479,0.67635,7.881241e-19,6.180827e-18,1.000000,7.109166e-17,6.788245e-13,1.000000,1.281257e-03,3.478004e-03,0.995241
22,1.0,sur01_02,surrogate1,0.546,0.304,0.779,0.479,0.67635,1.094118e-04,2.781850e-03,0.997109,9.853176e-05,6.534315e-25,0.999901,2.869741e-04,1.645950e-48,0.999713
19,1.0,sur00_02,surrogate0,0.545,0.304,0.779,0.479,0.67535,5.765699e-10,1.089045e-04,0.999891,7.029265e-17,1.206326e-03,0.998794,4.025079e-26,1.332360e-12,1.000000
5,1.0,prior04,init,0.543,0.305,0.788,0.473,0.67340,1.500000e-01,4.500000e-01,0.400000,2.000000e-01,2.000000e-01,0.600000,1.500000e-01,5.500000e-01,0.300000
44,2.0,sur00_02,surrogate0,0.546,0.305,0.725,0.502,0.67785,5.885692e-01,2.807420e-01,0.130689,2.807950e-01,6.562040e-02,0.653585,2.551199e-01,6.621916e-01,0.082688
42,2.0,sur00_00,surrogate0,0.546,0.304,0.724,0.503,0.67755,6.179328e-01,2.391548e-01,0.142912,1.681099e-01,5.772729e-02,0.774163,1.241891e-01,7.884281e-01,0.087383


report: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/02_mix_weight_optimizer/02_mix_weight_optimizer_report.md


## Optional Expanded Search

In [5]:
# Optional: turn this on after the first optimizer pass if the mini/full ranking looks stable.
RUN_EXPANDED_SEARCH = False

if RUN_EXPANDED_SEARCH:
    expanded_workspace = JUDGER_ROOT / "output" / "02_mix_weight_optimizer_expanded"
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    log_path = LOG_DIR / f"02_mix_weight_optimizer_expanded_{timestamp}.log"
    cmd = [
        sys.executable,
        str(RUNNER),
        "--workspace-root", str(expanded_workspace),
        "--rounds", "1,2,3,4,5",
        "--mini-images", "768",
        "--random-candidates", "12",
        "--surrogate-iterations", "3",
        "--surrogate-pool", "96",
        "--surrogate-evals", "4",
        "--full-eval-topk", "3",
        "--val-batch-size", "32",
        "--force",
    ]
    print(" ".join(cmd))
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.run(cmd, cwd=REPO_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        log.write(proc.stdout)
    print("returncode:", proc.returncode)
    print("log:", log_path)
    print(proc.stdout[-6000:])
    if proc.returncode != 0:
        raise SystemExit(proc.returncode)
